# Databricks Group Audit Tool

Audits a target group's **membership** and **Unity Catalog permissions** across all workspaces in your account.

## What it does
1. **Resolves group membership** recursively (nested groups, users, service principals)
2. **Discovers workspaces** from the Account API (or uses explicit URLs you provide)
3. **Scans catalog permissions** across workspaces with deduplication, classifying each grant as:
   - **Direct** — the group itself has the grant
   - **Upstream** — a parent group of the target has the grant
   - **Member Direct** — an individual user/SP in the group holds a personal grant
4. **Optionally scans schema-level** and **table/view-level** permissions within accessible catalogs
5. **Detects redundant grants** — flags member privileges already covered by the group (full or partial overlap)
6. **Generates REVOKE SQL** — copy-paste-ready cleanup script for redundant grants
7. **Exports to Delta** — optionally persists every run with a timestamp for historical drift tracking

## Inputs
| Widget | Description |
|---|---|
| `secret_scope` | *(optional)* Databricks secret scope containing `client_id`, `client_secret`, and `account_id` keys. When set, secrets take priority over the plain-text widgets below. |
| `client_id` | Service Principal application (client) ID |
| `client_secret` | Service Principal secret |
| `account_id` | Databricks Account ID. **Auto-detected** from the current workspace if left blank. |
| `cloud_provider` | `azure` / `aws` / `gcp` — selects the correct account console endpoint |
| `target_group` | Display name of the group to audit |
| `workspace_urls` | *(optional)* Comma-separated workspace URLs; leave empty to scan all |
| `scan_schemas` | `true` / `false` — whether to drill into schema-level grants |
| `scan_tables` | `true` / `false` — whether to drill into table/view-level grants (implies schema scan) |
| `export_delta_path` | *(optional)* Delta path (e.g. `dbfs:/audits/group_audit`). When set, all result DataFrames are appended with a timestamp column for historical comparison. |

> **Tip — recommended setup:** store `client_id`, `client_secret`, and `account_id` in a secret scope (Databricks-backed or Azure Key Vault‑backed) and only fill in the `secret_scope` widget.

## Outputs
| DataFrame | Contents |
|---|---|
| `df_permissions` | All catalog-level grants |
| `df_membership` | Users & SPs with inheritance paths |
| `df_inheritance` | Permission source trace (direct / upstream / personal) |
| `df_redundancy` | Overlap analysis with revoke recommendations |
| `df_schema_grants` | Schema-level grants (when enabled) |
| `df_table_grants` | Table/view-level grants (when enabled) |
| `revoke_sql` | Auto-generated REVOKE SQL cleanup script |

In [0]:
# ---------------------------------------------------------------------------
# Detect whether the pip-installable package is available.
# When installed, all class definitions below are skipped and the package
# versions are used directly.  When not installed, the notebook falls back
# to self-contained inline definitions (no external deps beyond requests).
# ---------------------------------------------------------------------------
PACKAGE_AVAILABLE = False

try:
    from databricks_group_audit import (
        DatabricksAPIClient,
        GroupMembershipResolver,
        WorkspaceDiscovery,
        CatalogPermissionScanner,
        SchemaPermissionScanner,
        TablePermissionScanner,
        RedundancyDetector,
        RevokeScriptGenerator,
        # models
        MemberType, GroupMember, GroupNode, WorkspaceInfo,
        GrantSource, CatalogGrant, SchemaGrant, TableGrant,
        RedundancyLevel, RedundancyResult,
        WORKSPACE_DOMAIN_MAP,
        classify_grant, build_member_lookups,
    )
    PACKAGE_AVAILABLE = True
    import databricks_group_audit as _pkg
    print(f"\u2705 Using installed databricks-group-audit package (v{_pkg.__version__})")
    print("   Class definition cells will be skipped.")
except ImportError:
    print("\U0001f4e6 Package not installed \u2014 using inline definitions below.")
    print("   Install with: %pip install databricks-group-audit")

In [0]:
import json, requests as _req

# ---------------------------------------------------------------------------
# Widget defaults
# ---------------------------------------------------------------------------
_defaults = {
    "secret_scope": "",
    "client_id": "",
    "client_secret": "",
    "account_id": "",
    "target_group": "",
    "workspace_urls": "",
    "export_delta_path": "",
    "max_retries": "5",
    "retry_base_delay": "1.0",
    "retry_max_delay": "60.0",
}

for _k, _v in _defaults.items():
    try:
        dbutils.widgets.text(_k, _v)
    except Exception:
        pass

try:
    dbutils.widgets.dropdown("cloud_provider", "azure", ["azure", "aws", "gcp"], "Cloud Provider")
    dbutils.widgets.dropdown("scan_schemas", "false", ["true", "false"], "Scan Schema Permissions")
    dbutils.widgets.dropdown("scan_tables", "false", ["true", "false"], "Scan Table Permissions")
except Exception:
    pass

# ---------------------------------------------------------------------------
# Resolve credentials: secret scope takes priority over plain-text widgets
# ---------------------------------------------------------------------------
SECRET_SCOPE = dbutils.widgets.get("secret_scope").strip()

def _secret_or_widget(key: str) -> str:
    """Return value from secret scope if configured, otherwise from widget."""
    if SECRET_SCOPE:
        try:
            return dbutils.secrets.get(scope=SECRET_SCOPE, key=key)
        except Exception:
            pass
    return dbutils.widgets.get(key).strip()

CLIENT_ID         = _secret_or_widget("client_id")
CLIENT_SECRET     = _secret_or_widget("client_secret")
ACCOUNT_ID        = _secret_or_widget("account_id")
TARGET_GROUP_NAME = dbutils.widgets.get("target_group").strip()
WORKSPACE_URLS    = dbutils.widgets.get("workspace_urls").strip()
CLOUD_PROVIDER    = dbutils.widgets.get("cloud_provider").strip().lower()
SCAN_SCHEMAS      = dbutils.widgets.get("scan_schemas").lower() == "true"
SCAN_TABLES       = dbutils.widgets.get("scan_tables").lower() == "true"
EXPORT_DELTA_PATH = dbutils.widgets.get("export_delta_path").strip()

# Rate-limit / retry tuning
MAX_RETRIES      = int(dbutils.widgets.get("max_retries"))
RETRY_BASE_DELAY = float(dbutils.widgets.get("retry_base_delay"))
RETRY_MAX_DELAY  = float(dbutils.widgets.get("retry_max_delay"))

# ---------------------------------------------------------------------------
# Resolve account console host per cloud
# ---------------------------------------------------------------------------
ACCOUNT_HOST_MAP = {
    "azure": "https://accounts.azuredatabricks.net",
    "aws":   "https://accounts.cloud.databricks.com",
    "gcp":   "https://accounts.gcp.databricks.com",
}
ACCOUNT_HOST = ACCOUNT_HOST_MAP.get(CLOUD_PROVIDER, ACCOUNT_HOST_MAP["azure"])

# ---------------------------------------------------------------------------
# Auto-detect Account ID when not explicitly provided
# ---------------------------------------------------------------------------
if not ACCOUNT_ID:
    try:
        _host = spark.conf.get("spark.databricks.workspaceUrl")
        _token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().getOrElse(None)
        if _host and _token:
            _r = _req.get(
                f"https://{_host}/api/2.0/preview/accounts/current",
                headers={"Authorization": f"Bearer {_token}"},
                timeout=10,
            )
            if _r.ok:
                ACCOUNT_ID = _r.json().get("account_id", "")
    except Exception:
        pass

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
_src = f"secret scope '{SECRET_SCOPE}'" if SECRET_SCOPE else "widgets"
print(f"Configuration loaded (credentials from {_src}):")
print(f"  Cloud Provider: {CLOUD_PROVIDER.upper()}")
print(f"  Account Host:   {ACCOUNT_HOST}")
print(f"  Account ID:     {ACCOUNT_ID[:8]}..." if ACCOUNT_ID else "  Account ID:     (not set)")
print(f"  Client ID:      {CLIENT_ID[:8]}..." if CLIENT_ID else "  Client ID:      (not set)")
print(f"  Target Group:   {TARGET_GROUP_NAME}")
print(f"  Workspace URLs: {WORKSPACE_URLS if WORKSPACE_URLS else '(all workspaces)'}")
print(f"  Scan Schemas:   {SCAN_SCHEMAS}")
print(f"  Scan Tables:    {SCAN_TABLES}")
print(f"  Export Delta:   {EXPORT_DELTA_PATH if EXPORT_DELTA_PATH else '(disabled)'}")
print(f"  Retry Config:   max_retries={MAX_RETRIES}, base_delay={RETRY_BASE_DELAY}s, max_delay={RETRY_MAX_DELAY}s")


In [0]:
if not PACKAGE_AVAILABLE:
    import requests
    import time
    import json
    from typing import Optional, Dict, Any, List
    from functools import wraps
    from dataclasses import dataclass, field
    from datetime import datetime, timedelta
    import threading

    @dataclass
    class TokenCache:
        """Thread-safe token cache with expiry."""
        token: Optional[str] = None
        expires_at: Optional[datetime] = None
        lock: threading.Lock = field(default_factory=threading.Lock)
        
        def get_token(self) -> Optional[str]:
            with self.lock:
                if self.token and self.expires_at and datetime.now() < self.expires_at:
                    return self.token
                return None
        
        def set_token(self, token: str, expires_in: int):
            with self.lock:
                self.token = token
                self.expires_at = datetime.now() + timedelta(seconds=expires_in - 60)

    class DatabricksAPIClient:
        """HTTP client with OAuth authentication and automatic retry on 429/5xx.
        
        Supports Azure, AWS, and GCP account console hosts.
        """
        
        SCIM_PAGE_SIZE = 100
        
        def __init__(self, client_id: str, client_secret: str, account_id: str,
                     account_host: str = "https://accounts.azuredatabricks.net",
                     max_retries: int = 5, base_delay: float = 1.0, max_delay: float = 60.0):
            self.client_id = client_id
            self.client_secret = client_secret
            self.account_id = account_id
            self.max_retries = max_retries
            self.base_delay = base_delay
            self.max_delay = max_delay
            self._account_token_cache = TokenCache()
            self._workspace_token_caches: Dict[str, TokenCache] = {}
            self.account_host = account_host
            self.session = requests.Session()
        
        # -- OAuth token helpers ------------------------------------------------
        def _get_oauth_token(self, host: str, scope: Optional[str] = None) -> str:
            token_url = f"{host}/oidc/v1/token"
            data = {
                "grant_type": "client_credentials",
                "client_id": self.client_id,
                "client_secret": self.client_secret,
            }
            if scope:
                data["scope"] = scope
            response = self.session.post(token_url, data=data)
            response.raise_for_status()
            token_data = response.json()
            return token_data["access_token"], token_data.get("expires_in", 3600)
        
        def _get_account_token(self) -> str:
            cached = self._account_token_cache.get_token()
            if cached:
                return cached
            token, expires_in = self._get_oauth_token(self.account_host, scope="all-apis")
            self._account_token_cache.set_token(token, expires_in)
            return token
        
        def _get_workspace_token(self, workspace_host: str) -> str:
            if workspace_host not in self._workspace_token_caches:
                self._workspace_token_caches[workspace_host] = TokenCache()
            cache = self._workspace_token_caches[workspace_host]
            cached = cache.get_token()
            if cached:
                return cached
            token, expires_in = self._get_oauth_token(workspace_host, scope="all-apis")
            cache.set_token(token, expires_in)
            return token
        
        # -- Retry engine ------------------------------------------------------
        def _request_with_retry(self, method: str, url: str, token: str, **kwargs) -> requests.Response:
            headers = kwargs.pop("headers", {})
            headers["Authorization"] = f"Bearer {token}"
            headers["Content-Type"] = "application/json"
            
            last_exception = None
            for attempt in range(self.max_retries + 1):
                try:
                    response = self.session.request(method, url, headers=headers, **kwargs)
                    if response.status_code < 400 or (400 <= response.status_code < 500 and response.status_code != 429):
                        return response
                    if response.status_code == 429 or response.status_code >= 500:
                        if attempt < self.max_retries:
                            retry_after = response.headers.get("Retry-After")
                            if retry_after:
                                delay = float(retry_after)
                            else:
                                delay = min(self.base_delay * (2 ** attempt), self.max_delay)
                                delay = delay * (0.5 + 0.5 * (hash(str(time.time())) % 100) / 100)
                            print(f"  \u23f3 {response.status_code} - retrying in {delay:.1f}s ({attempt + 1}/{self.max_retries})")
                            time.sleep(delay)
                            continue
                    return response
                except requests.exceptions.RequestException as e:
                    last_exception = e
                    if attempt < self.max_retries:
                        delay = min(self.base_delay * (2 ** attempt), self.max_delay)
                        print(f"  \u23f3 Request failed ({e}), retrying in {delay:.1f}s ({attempt + 1}/{self.max_retries})")
                        time.sleep(delay)
                        continue
                    raise
            if last_exception:
                raise last_exception
            return response
        
        # -- High-level API methods --------------------------------------------
        def account_api(self, method: str, endpoint: str, **kwargs) -> Dict[str, Any]:
            token = self._get_account_token()
            url = f"{self.account_host}/api/2.0/accounts/{self.account_id}{endpoint}"
            response = self._request_with_retry(method, url, token, **kwargs)
            response.raise_for_status()
            return response.json() if response.text else {}
        
        def workspace_api(self, workspace_host: str, method: str, endpoint: str, **kwargs) -> Dict[str, Any]:
            if not workspace_host.startswith("https://"):
                workspace_host = f"https://{workspace_host}"
            token = self._get_workspace_token(workspace_host)
            url = f"{workspace_host}{endpoint}"
            response = self._request_with_retry(method, url, token, **kwargs)
            response.raise_for_status()
            return response.json() if response.text else {}
        
        # -- Paginated SCIM helpers --------------------------------------------
        def scim_list_all(self, resource: str, params: Optional[Dict] = None) -> List[Dict]:
            """Paginate through a SCIM list endpoint (Groups, Users, ServicePrincipals)."""
            all_resources: List[Dict] = []
            start_index = 1
            while True:
                page_params = {"startIndex": start_index, "count": self.SCIM_PAGE_SIZE}
                if params:
                    page_params.update(params)
                response = self.account_api("GET", f"/scim/v2/{resource}", params=page_params)
                resources = response.get("Resources", [])
                all_resources.extend(resources)
                total_results = response.get("totalResults", len(all_resources))
                items_per_page = response.get("itemsPerPage", len(resources))
                if not resources or len(all_resources) >= total_results:
                    break
                start_index += items_per_page
            return all_resources

# Initialize the API client (cloud-aware, tunable retry)
api_client = DatabricksAPIClient(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    account_id=ACCOUNT_ID,
    account_host=ACCOUNT_HOST,
    max_retries=MAX_RETRIES,
    base_delay=RETRY_BASE_DELAY,
    max_delay=RETRY_MAX_DELAY,
)

print(f"\u2705 API client initialized ({CLOUD_PROVIDER.upper()} \u2192 {ACCOUNT_HOST})")

In [0]:
if not PACKAGE_AVAILABLE:
    import logging
    from dataclasses import dataclass, field
    from typing import Set, List, Dict, Optional
    from enum import Enum

    log = logging.getLogger(__name__)
    MAX_RECURSION_DEPTH = 50

    class MemberType(Enum):
        USER = "User"
        SERVICE_PRINCIPAL = "ServicePrincipal"
        GROUP = "Group"

    @dataclass
    class GroupMember:
        """Represents a member of a group."""
        id: str
        display_name: str
        member_type: MemberType
        email: Optional[str] = None
        application_id: Optional[str] = None
        parent_groups: List[str] = field(default_factory=list)

    @dataclass
    class GroupNode:
        """Represents a group in the hierarchy tree."""
        id: str
        display_name: str
        direct_users: List[GroupMember] = field(default_factory=list)
        direct_service_principals: List[GroupMember] = field(default_factory=list)
        nested_groups: Dict[str, 'GroupNode'] = field(default_factory=dict)
        parent_path: List[str] = field(default_factory=list)

    class GroupMembershipResolver:
        """Walk the SCIM API to build a full group hierarchy tree."""
        
        def __init__(self, api_client: DatabricksAPIClient):
            self.api_client = api_client
            self._group_cache: Dict[str, Dict] = {}
            self._user_cache: Dict[str, Dict] = {}
            self._sp_cache: Dict[str, Dict] = {}
            self._resolved_groups: Set[str] = set()
        
        def clear_caches(self) -> None:
            """Reset all caches (useful between separate audit runs)."""
            self._group_cache.clear()
            self._user_cache.clear()
            self._sp_cache.clear()
            self._resolved_groups.clear()
        
        # -- SCIM helpers ---------------------------------------------------
        def _get_group_by_name(self, group_name: str) -> Optional[Dict]:
            try:
                response = self.api_client.account_api(
                    "GET", "/scim/v2/Groups",
                    params={"filter": f'displayName eq "{group_name}"'}
                )
                resources = response.get("Resources", [])
                return resources[0] if resources else None
            except Exception as exc:
                log.warning("Failed to fetch group '%s': %s", group_name, exc)
                print(f"  \u26a0\ufe0f Failed to fetch group '{group_name}': {exc}")
                return None
        
        def _get_group_by_id(self, group_id: str) -> Optional[Dict]:
            if group_id in self._group_cache:
                return self._group_cache[group_id]
            try:
                response = self.api_client.account_api("GET", f"/scim/v2/Groups/{group_id}")
                self._group_cache[group_id] = response
                return response
            except Exception as exc:
                log.warning("Failed to fetch group id '%s': %s", group_id, exc)
                print(f"  \u26a0\ufe0f Failed to fetch group ID '{group_id}': {exc}")
                return None
        
        def _get_user_by_id(self, user_id: str) -> Optional[Dict]:
            if user_id in self._user_cache:
                return self._user_cache[user_id]
            try:
                response = self.api_client.account_api("GET", f"/scim/v2/Users/{user_id}")
                self._user_cache[user_id] = response
                return response
            except Exception as exc:
                log.warning("Failed to fetch user id '%s': %s", user_id, exc)
                print(f"  \u26a0\ufe0f Failed to fetch user ID '{user_id}': {exc}")
                return None
        
        def _get_sp_by_id(self, sp_id: str) -> Optional[Dict]:
            if sp_id in self._sp_cache:
                return self._sp_cache[sp_id]
            try:
                response = self.api_client.account_api("GET", f"/scim/v2/ServicePrincipals/{sp_id}")
                self._sp_cache[sp_id] = response
                return response
            except Exception as exc:
                log.warning("Failed to fetch SP id '%s': %s", sp_id, exc)
                print(f"  \u26a0\ufe0f Failed to fetch SP ID '{sp_id}': {exc}")
                return None
        
        # -- Bulk pre-fetch ------------------------------------------------
        def _prefetch_users_and_sps(self) -> None:
            """Bulk-fetch all users and SPs into caches (two paginated calls).
            
            Trades a wider initial fetch for zero per-member API calls.
            For accounts with <10,000 users this is significantly faster than N+1.
            """
            if not self._user_cache:
                try:
                    for u in self.api_client.scim_list_all("Users"):
                        self._user_cache[u.get("id", "")] = u
                    log.info("Pre-fetched %d users", len(self._user_cache))
                    print(f"  \U0001f4e5 Pre-fetched {len(self._user_cache)} users")
                except Exception as exc:
                    log.warning("Bulk user fetch failed, falling back to per-member: %s", exc)
                    print(f"  \u26a0\ufe0f Bulk user fetch failed, falling back to per-member: {exc}")
            
            if not self._sp_cache:
                try:
                    for sp in self.api_client.scim_list_all("ServicePrincipals"):
                        self._sp_cache[sp.get("id", "")] = sp
                    log.info("Pre-fetched %d service principals", len(self._sp_cache))
                    print(f"  \U0001f4e5 Pre-fetched {len(self._sp_cache)} service principals")
                except Exception as exc:
                    log.warning("Bulk SP fetch failed, falling back to per-member: %s", exc)
                    print(f"  \u26a0\ufe0f Bulk SP fetch failed, falling back to per-member: {exc}")
        
        # -- Recursive resolver --------------------------------------------
        def _resolve_group_recursive(self, group_id: str, parent_path: List[str] = None, depth: int = 0) -> GroupNode:
            if parent_path is None:
                parent_path = []
            
            if depth > MAX_RECURSION_DEPTH:
                log.warning("Max recursion depth (%d) reached at group '%s'", MAX_RECURSION_DEPTH, group_id)
                print(f"  \u26a0\ufe0f Max recursion depth ({MAX_RECURSION_DEPTH}) reached at group '{group_id}'")
                return None
            
            if group_id in self._resolved_groups:
                print(f"  {'  ' * depth}\u21a9\ufe0f Skipping already-resolved group: {group_id}")
                return None
            self._resolved_groups.add(group_id)
            group_data = self._get_group_by_id(group_id)
            if not group_data:
                return None
            group_name = group_data.get("displayName", group_id)
            print(f"  {'  ' * depth}\U0001f4c2 Resolving group: {group_name}")
            node = GroupNode(id=group_id, display_name=group_name, parent_path=list(parent_path))
            members = group_data.get("members", [])
            current_path = parent_path + [group_name]
            for member in members:
                ref = member.get("$ref", "")
                member_value = member.get("value")
                member_display = member.get("display", "Unknown")
                if "/Users/" in ref:
                    user_data = self._get_user_by_id(member_value)
                    email = None
                    if user_data:
                        emails = user_data.get("emails", [])
                        email = emails[0].get("value") if emails else None
                        member_display = user_data.get("displayName", member_display)
                    node.direct_users.append(GroupMember(
                        id=member_value, display_name=member_display,
                        member_type=MemberType.USER, email=email,
                        parent_groups=list(current_path)
                    ))
                    print(f"  {'  ' * (depth + 1)}\U0001f464 User: {member_display}")
                elif "/ServicePrincipals/" in ref:
                    sp_data = self._get_sp_by_id(member_value)
                    app_id = None
                    if sp_data:
                        app_id = sp_data.get("applicationId")
                        member_display = sp_data.get("displayName", member_display)
                    node.direct_service_principals.append(GroupMember(
                        id=member_value, display_name=member_display,
                        member_type=MemberType.SERVICE_PRINCIPAL, application_id=app_id,
                        parent_groups=list(current_path)
                    ))
                    print(f"  {'  ' * (depth + 1)}\U0001f916 Service Principal: {member_display}")
                elif "/Groups/" in ref:
                    nested_node = self._resolve_group_recursive(member_value, current_path, depth + 1)
                    if nested_node:
                        node.nested_groups[member_value] = nested_node
            return node
        
        # -- Public interface ----------------------------------------------
        def resolve_group(self, group_name: str) -> Optional[GroupNode]:
            """Resolve full membership hierarchy for a group by display name."""
            print(f"\n\U0001f50d Resolving membership for group: {group_name}")
            self._resolved_groups.clear()
            self._prefetch_users_and_sps()
            group_data = self._get_group_by_name(group_name)
            if not group_data:
                print(f"  \u274c Group '{group_name}' not found")
                return None
            return self._resolve_group_recursive(group_data["id"])
        
        @staticmethod
        def get_all_members_flat(node: GroupNode) -> Dict[str, List[GroupMember]]:
            """Flatten the hierarchy into deduplicated user and SP lists."""
            users: Dict[str, GroupMember] = {}
            sps: Dict[str, GroupMember] = {}
            def _collect(n: GroupNode):
                for u in n.direct_users:
                    if u.id not in users:
                        users[u.id] = u
                for sp in n.direct_service_principals:
                    if sp.id not in sps:
                        sps[sp.id] = sp
                for nested in n.nested_groups.values():
                    _collect(nested)
            _collect(node)
            return {"users": list(users.values()), "service_principals": list(sps.values())}

# Initialize the resolver
group_resolver = GroupMembershipResolver(api_client)
print("\u2705 Group membership resolver initialized")

In [0]:
if not PACKAGE_AVAILABLE:
    # Workspace URL suffix per cloud provider
    WORKSPACE_DOMAIN_MAP = {
        "AZURE": ".azuredatabricks.net",
        "AWS":   ".cloud.databricks.com",
        "GCP":   ".gcp.databricks.com",
    }

    @dataclass
    class WorkspaceInfo:
        """Information about a Databricks workspace."""
        workspace_id: str
        deployment_name: str
        workspace_name: str
        workspace_url: str
        cloud: str
        region: str

    class WorkspaceDiscovery:
        """Discovers workspaces from Account API or uses provided URLs."""
        
        def __init__(self, api_client: DatabricksAPIClient, cloud_provider: str = "AZURE"):
            self.api_client = api_client
            self.cloud_provider = cloud_provider.upper()
        
        def _build_workspace_url(self, deployment_name: str, cloud: str = None) -> str:
            """Construct workspace URL from deployment name and cloud provider.
            Only used as a fallback when the API does not return a direct URL.
            """
            cloud = (cloud or self.cloud_provider).upper()
            domain = WORKSPACE_DOMAIN_MAP.get(cloud, WORKSPACE_DOMAIN_MAP["AZURE"])
            return f"https://{deployment_name}{domain}"
        
        def _resolve_workspace_url(self, ws: dict, cloud: str) -> str:
            """Resolve the actual workspace URL.
            Prefers deployment_url from Account API (canonical for AWS
            adb-<id> style workspaces). Falls back to deployment_name.
            """
            api_url = ws.get("deployment_url", "").strip()
            if api_url:
                if not api_url.startswith("https://"):
                    api_url = f"https://{api_url}"
                return api_url.rstrip("/")
            deployment_name = ws.get("deployment_name", "")
            if deployment_name:
                return self._build_workspace_url(deployment_name, cloud)
            # Last resort: reconstruct from workspace_id for AWS
            if cloud == "AWS":
                wid = ws.get("workspace_id", "")
                if wid:
                    return f"https://adb-{wid}.cloud.databricks.com"
            return ""
        
        def get_all_workspaces(self) -> List[WorkspaceInfo]:
            """Fetch all workspaces from the Account API."""
            print("\n\U0001f310 Fetching all workspaces from Account API...")
            try:
                response = self.api_client.account_api("GET", "/workspaces")
                workspaces = []
                for ws in response:
                    workspace_id = str(ws.get("workspace_id", ""))
                    deployment_name = ws.get("deployment_name", "")
                    workspace_name = ws.get("workspace_name", deployment_name)
                    cloud = ws.get("cloud", self.cloud_provider).upper()
                    workspace_url = self._resolve_workspace_url(ws, cloud)
                    if cloud == "AZURE":
                        region = ws.get("azure_workspace_info", {}).get("region", "unknown")
                    elif cloud == "GCP":
                        region = ws.get("gcp_workspace_info", {}).get("region", ws.get("cloud_region", "unknown"))
                    else:
                        region = ws.get("aws_region", ws.get("cloud_region", "unknown"))
                    workspaces.append(WorkspaceInfo(
                        workspace_id=workspace_id, deployment_name=deployment_name,
                        workspace_name=workspace_name, workspace_url=workspace_url,
                        cloud=cloud, region=region
                    ))
                    print(f"  \U0001f4bc {workspace_name} ({workspace_url})")
                print(f"\n\u2705 Found {len(workspaces)} workspace(s)")
                return workspaces
            except Exception as e:
                print(f"  \u274c Error fetching workspaces: {e}")
                return []
        
        def parse_workspace_urls(self, urls_str: str) -> List[WorkspaceInfo]:
            """Parse comma-separated workspace URLs into WorkspaceInfo objects."""
            if not urls_str.strip():
                return []
            workspaces = []
            for url in urls_str.split(","):
                url = url.strip()
                if not url:
                    continue
                if not url.startswith("https://"):
                    url = f"https://{url}"
                host = url.replace("https://", "").split("/")[0]
                detected_cloud = self.cloud_provider
                deployment_name = host
                for cloud_key, domain in WORKSPACE_DOMAIN_MAP.items():
                    if host.endswith(domain):
                        detected_cloud = cloud_key
                        deployment_name = host.replace(domain, "")
                        break
                workspaces.append(WorkspaceInfo(
                    workspace_id="manual", deployment_name=deployment_name,
                    workspace_name=deployment_name, workspace_url=url.rstrip("/"),
                    cloud=detected_cloud, region="unknown"
                ))
            return workspaces
        
        def discover(self, explicit_urls: str = "") -> List[WorkspaceInfo]:
            if explicit_urls.strip():
                print(f"\n\U0001f310 Using explicitly provided workspace URLs...")
                workspaces = self.parse_workspace_urls(explicit_urls)
                for ws in workspaces:
                    print(f"  \U0001f4bc {ws.workspace_name} ({ws.workspace_url})")
                print(f"\n\u2705 Using {len(workspaces)} workspace(s)")
                return workspaces
            else:
                return self.get_all_workspaces()

# Initialize workspace discovery (cloud-aware)
workspace_discovery = WorkspaceDiscovery(api_client, cloud_provider=CLOUD_PROVIDER)
print("\u2705 Workspace discovery initialized")

In [0]:
if not PACKAGE_AVAILABLE:
    from enum import Enum
    from typing import Tuple

    class GrantSource(Enum):
        """How a permission was granted."""
        DIRECT = "Direct"
        UPSTREAM = "Upstream"
        MEMBER_DIRECT = "Member Direct"

    @dataclass
    class CatalogGrant:
        """Represents a permission grant on a catalog."""
        catalog_name: str
        workspace_name: str
        workspace_url: str
        principal: str
        principal_type: str
        privileges: List[str]
        grant_source: GrantSource
        inherited_from: Optional[str] = None
        member_of_target: bool = False

    # -----------------------------------------------------------------
    # Shared classification helpers (used by schema/table scanners too)
    # -----------------------------------------------------------------
    def build_member_lookups(all_members):
        """Build fast-lookup sets from the flat member lists."""
        member_emails = {m.email.lower() for m in all_members["users"] if m.email}
        member_names = {m.display_name for m in all_members["users"]}
        sp_names = {sp.display_name for sp in all_members["service_principals"]}
        sp_app_ids = {sp.application_id for sp in all_members["service_principals"] if sp.application_id}
        return member_emails, member_names, sp_names, sp_app_ids

    def classify_grant(principal, target_group_name, upstream_groups,
                       member_emails, member_names, sp_names, sp_app_ids):
        """Classify a grant principal into Direct / Upstream / Member Direct.
        Returns (grant_source, principal_type, inherited_from, member_of_target)
        or None if unrelated.
        """
        if principal == target_group_name:
            return GrantSource.DIRECT, "GROUP", None, False
        if principal in upstream_groups:
            return GrantSource.UPSTREAM, "GROUP", principal, False
        p_lower = principal.lower()
        p_clean = principal.replace("`", "")
        if p_lower in member_emails or principal in member_names or p_clean in member_names:
            return GrantSource.MEMBER_DIRECT, "USER", None, True
        if principal in sp_names or principal in sp_app_ids or p_clean in sp_names:
            return GrantSource.MEMBER_DIRECT, "SERVICE_PRINCIPAL", None, True
        return None

    class CatalogPermissionScanner:
        """Scans catalog permissions across workspaces with deduplication."""
        
        def __init__(self, api_client: DatabricksAPIClient, group_resolver: GroupMembershipResolver):
            self.api_client = api_client
            self.group_resolver = group_resolver
            self._scanned_catalogs: Set[str] = set()
        
        def _get_catalogs(self, workspace: WorkspaceInfo) -> List[Dict]:
            try:
                response = self.api_client.workspace_api(
                    workspace.workspace_url, "GET", "/api/2.1/unity-catalog/catalogs"
                )
                return response.get("catalogs", [])
            except Exception as e:
                print(f"    \u26a0\ufe0f Error listing catalogs: {e}")
                return []
        
        def _get_catalog_grants(self, workspace: WorkspaceInfo, catalog_name: str) -> List[Dict]:
            try:
                response = self.api_client.workspace_api(
                    workspace.workspace_url, "GET",
                    f"/api/2.1/unity-catalog/permissions/catalog/{catalog_name}"
                )
                return response.get("privilege_assignments", [])
            except Exception as e:
                print(f"    \u26a0\ufe0f Error getting grants for catalog {catalog_name}: {e}")
                return []
        
        def _get_groups_containing_target(self, target_group_name: str) -> Dict[str, str]:
            """Find ALL upstream groups -- recursive parents of the target.
            Walks upward through the group membership graph via BFS so that
            transitive ancestors are captured (A contains B contains target
            -> both A and B are returned).
            """
            all_groups = self.api_client.scim_list_all("Groups")
            id_to_name = {}
            child_to_parents = {}
            target_id = None
            for g in all_groups:
                gid = g.get("id")
                gname = g.get("displayName", "")
                id_to_name[gid] = gname
                if gname == target_group_name:
                    target_id = gid
                for m in g.get("members", []):
                    child_id = m.get("value")
                    if child_id:
                        child_to_parents.setdefault(child_id, set()).add(gid)
            if not target_id:
                return {}
            # BFS upward from target
            upstream = {}
            queue = [target_id]
            visited = {target_id}
            while queue:
                current = queue.pop(0)
                for parent_id in child_to_parents.get(current, set()):
                    if parent_id not in visited:
                        visited.add(parent_id)
                        upstream[id_to_name.get(parent_id, parent_id)] = parent_id
                        queue.append(parent_id)
            return upstream
        
        def _classify_grant(self, principal, privileges, catalog_name, workspace,
                            target_group_name, upstream_groups,
                            member_emails, member_names, sp_names, sp_app_ids):
            """Classify a single catalog grant."""
            result = classify_grant(
                principal, target_group_name, upstream_groups,
                member_emails, member_names, sp_names, sp_app_ids,
            )
            if result is None:
                return None
            source, ptype, inherited, member = result
            return CatalogGrant(
                catalog_name=catalog_name,
                workspace_name=workspace.workspace_name,
                workspace_url=workspace.workspace_url,
                principal=principal, principal_type=ptype,
                privileges=privileges, grant_source=source,
                inherited_from=inherited, member_of_target=member,
            )
        
        def scan_workspace(self, workspace, target_group_name, group_node, all_members):
            print(f"\n\U0001f50d Scanning workspace: {workspace.workspace_name}")
            grants = []
            catalogs = self._get_catalogs(workspace)
            upstream_groups = self._get_groups_containing_target(target_group_name)
            member_emails, member_names, sp_names, sp_app_ids = build_member_lookups(all_members)
            for catalog in catalogs:
                catalog_name = catalog.get("name", "")
                if catalog_name in self._scanned_catalogs:
                    print(f"  \u21a9\ufe0f Skipping {catalog_name} (already audited)")
                    continue
                self._scanned_catalogs.add(catalog_name)
                print(f"  \U0001f4da Scanning catalog: {catalog_name}")
                for grant in self._get_catalog_grants(workspace, catalog_name):
                    principal = grant.get("principal", "")
                    privileges = grant.get("privileges", [])
                    if not privileges:
                        continue
                    obj = self._classify_grant(
                        principal, privileges, catalog_name, workspace,
                        target_group_name, upstream_groups,
                        member_emails, member_names, sp_names, sp_app_ids
                    )
                    if obj:
                        tag = obj.grant_source.value.upper()
                        print(f"    {tag}: {principal} -> {privileges}")
                        grants.append(obj)
            return grants
        
        def scan_all_workspaces(self, workspaces, target_group_name, group_node, all_members):
            print(f"\n\U0001f680 Starting cross-workspace catalog permission scan...")
            print(f"   Target group: {target_group_name}")
            print(f"   Workspaces to scan: {len(workspaces)}")
            self._scanned_catalogs.clear()
            all_grants = []
            for workspace in workspaces:
                try:
                    all_grants.extend(self.scan_workspace(workspace, target_group_name, group_node, all_members))
                except Exception as e:
                    print(f"  \u274c Error scanning {workspace.workspace_name}: {e}")
            print(f"\n\u2705 Scan complete. {len(all_grants)} grant(s) across {len(self._scanned_catalogs)} catalog(s)")
            return all_grants

catalog_scanner = CatalogPermissionScanner(api_client, group_resolver)
print("\u2705 Catalog permission scanner initialized")

In [0]:
if not PACKAGE_AVAILABLE:
    @dataclass
    class SchemaGrant:
        """Represents a permission grant on a schema."""
        catalog_name: str
        schema_name: str
        workspace_name: str
        workspace_url: str
        principal: str
        principal_type: str
        privileges: List[str]
        grant_source: GrantSource
        inherited_from: Optional[str] = None
        member_of_target: bool = False

    class SchemaPermissionScanner:
        """Scans schema-level permissions within accessible catalogs."""
        
        def __init__(self, api_client: DatabricksAPIClient):
            self.api_client = api_client
        
        def _get_schemas(self, workspace: WorkspaceInfo, catalog_name: str) -> List[Dict]:
            try:
                response = self.api_client.workspace_api(
                    workspace.workspace_url, "GET",
                    f"/api/2.1/unity-catalog/schemas",
                    params={"catalog_name": catalog_name}
                )
                return response.get("schemas", [])
            except Exception as e:
                print(f"      \u26a0\ufe0f Error listing schemas in {catalog_name}: {e}")
                return []
        
        def _get_schema_grants(self, workspace, catalog_name, schema_name):
            try:
                response = self.api_client.workspace_api(
                    workspace.workspace_url, "GET",
                    f"/api/2.1/unity-catalog/permissions/schema/{catalog_name}.{schema_name}"
                )
                return response.get("privilege_assignments", [])
            except Exception:
                return []
        
        def scan_schemas(self, workspace, catalog_name, target_group_name,
                         all_members, upstream_groups):
            """Scan all schemas in a catalog for relevant permissions."""
            grants = []
            schemas = self._get_schemas(workspace, catalog_name)
            lookups = build_member_lookups(all_members)
            for schema in schemas:
                schema_name = schema.get("name", "")
                full_name = f"{catalog_name}.{schema_name}"
                schema_grants = self._get_schema_grants(workspace, catalog_name, schema_name)
                for grant in schema_grants:
                    principal = grant.get("principal", "")
                    privileges = grant.get("privileges", [])
                    if not privileges:
                        continue
                    result = classify_grant(
                        principal, target_group_name, upstream_groups, *lookups
                    )
                    if result is None:
                        continue
                    source, ptype, inherited, member = result
                    grant_obj = SchemaGrant(
                        catalog_name=catalog_name, schema_name=schema_name,
                        workspace_name=workspace.workspace_name,
                        workspace_url=workspace.workspace_url,
                        principal=principal, principal_type=ptype,
                        privileges=privileges, grant_source=source,
                        inherited_from=inherited, member_of_target=member,
                    )
                    grants.append(grant_obj)
                    tag = source.value.upper()
                    print(f"      \U0001f4c1 Schema {full_name}: {tag} {principal} -> {privileges}")
            return grants

# Initialize schema scanner
schema_scanner = SchemaPermissionScanner(api_client)
print("\u2705 Schema permission scanner initialized")

In [0]:
if not PACKAGE_AVAILABLE:
    @dataclass
    class TableGrant:
        """Represents a permission grant on a table or view."""
        catalog_name: str
        schema_name: str
        table_name: str
        full_name: str
        table_type: str
        workspace_name: str
        workspace_url: str
        principal: str
        principal_type: str
        privileges: List[str]
        grant_source: GrantSource
        inherited_from: Optional[str] = None
        member_of_target: bool = False

    class TablePermissionScanner:
        """Scans table/view-level permissions within accessible schemas."""
        
        def __init__(self, api_client: DatabricksAPIClient):
            self.api_client = api_client
        
        def _get_tables(self, workspace, catalog_name, schema_name):
            try:
                response = self.api_client.workspace_api(
                    workspace.workspace_url, "GET",
                    f"/api/2.1/unity-catalog/tables",
                    params={"catalog_name": catalog_name, "schema_name": schema_name}
                )
                return response.get("tables", [])
            except Exception as e:
                print(f"        \u26a0\ufe0f Error listing tables in {catalog_name}.{schema_name}: {e}")
                return []
        
        def _get_table_grants(self, workspace, full_table_name):
            try:
                response = self.api_client.workspace_api(
                    workspace.workspace_url, "GET",
                    f"/api/2.1/unity-catalog/permissions/table/{full_table_name}"
                )
                return response.get("privilege_assignments", [])
            except Exception:
                return []
        
        def scan_tables(self, workspace, catalog_name, schema_name,
                        target_group_name, all_members, upstream_groups):
            """Scan all tables in a schema for relevant permissions."""
            grants = []
            tables = self._get_tables(workspace, catalog_name, schema_name)
            lookups = build_member_lookups(all_members)
            for tbl in tables:
                tbl_name = tbl.get("name", "")
                tbl_type = tbl.get("table_type", "TABLE")
                full_name = f"{catalog_name}.{schema_name}.{tbl_name}"
                for grant in self._get_table_grants(workspace, full_name):
                    principal = grant.get("principal", "")
                    privileges = grant.get("privileges", [])
                    if not privileges:
                        continue
                    result = classify_grant(
                        principal, target_group_name, upstream_groups, *lookups
                    )
                    if result is None:
                        continue
                    source, ptype, inherited, member = result
                    obj = TableGrant(
                        catalog_name=catalog_name, schema_name=schema_name,
                        table_name=tbl_name, full_name=full_name,
                        table_type=tbl_type,
                        workspace_name=workspace.workspace_name,
                        workspace_url=workspace.workspace_url,
                        principal=principal, principal_type=ptype,
                        privileges=privileges, grant_source=source,
                        inherited_from=inherited, member_of_target=member,
                    )
                    grants.append(obj)
                    tag = source.value.upper()
                    print(f"        \U0001f4dd {full_name}: {tag} {principal} -> {privileges}")
            return grants

# Initialize table scanner
table_scanner = TablePermissionScanner(api_client)
print("\u2705 Table permission scanner initialized")

In [0]:
if not PACKAGE_AVAILABLE:
    from enum import Enum

    class RedundancyLevel(Enum):
        NONE = "None"
        PARTIAL = "Partial"
        FULL = "Full"

    @dataclass
    class RedundancyResult:
        catalog_name: str
        principal: str
        principal_type: str
        member_privileges: List[str]
        group_effective_privileges: List[str]
        redundant_privileges: List[str]
        additional_privileges: List[str]
        redundancy_level: RedundancyLevel
        recommendation: str

    class RedundancyDetector:
        """Detects redundant/overlapping grants between members and group."""
        
        PRIVILEGE_HIERARCHY = {
            "ALL_PRIVILEGES": [
                "USE_CATALOG", "USE_SCHEMA", "SELECT", "MODIFY", "CREATE_SCHEMA",
                "CREATE_TABLE", "CREATE_VIEW", "CREATE_FUNCTION", "CREATE_MODEL",
                "EXECUTE", "READ_VOLUME", "WRITE_VOLUME", "CREATE_VOLUME",
                "REFRESH", "APPLY_TAG", "BROWSE"
            ],
            "MODIFY": ["SELECT"],
            "WRITE_VOLUME": ["READ_VOLUME"],
            "CREATE_TABLE": ["USE_SCHEMA"],
            "CREATE_VIEW": ["USE_SCHEMA"],
            "CREATE_FUNCTION": ["USE_SCHEMA"],
            "CREATE_MODEL": ["USE_SCHEMA"],
            "CREATE_VOLUME": ["USE_SCHEMA"],
            "CREATE_SCHEMA": ["USE_CATALOG"],
        }
        
        def _expand_privileges(self, privileges):
            expanded = set(privileges)
            changed = True
            while changed:
                changed = False
                for priv in list(expanded):
                    implied = self.PRIVILEGE_HIERARCHY.get(priv, [])
                    for imp in implied:
                        if imp not in expanded:
                            expanded.add(imp)
                            changed = True
            return expanded
        
        def detect_redundancy(self, catalog_grants, target_group_name):
            print("\n\U0001f50d Detecting redundant/overlapping grants...")
            results = []
            grants_by_catalog = {}
            for grant in catalog_grants:
                if grant.catalog_name not in grants_by_catalog:
                    grants_by_catalog[grant.catalog_name] = []
                grants_by_catalog[grant.catalog_name].append(grant)
            for catalog_name, grants in grants_by_catalog.items():
                group_privileges = set()
                for grant in grants:
                    if grant.grant_source in [GrantSource.DIRECT, GrantSource.UPSTREAM]:
                        group_privileges.update(grant.privileges)
                group_effective = self._expand_privileges(list(group_privileges))
                for grant in grants:
                    if grant.grant_source != GrantSource.MEMBER_DIRECT:
                        continue
                    member_privs = set(grant.privileges)
                    redundant = member_privs & group_effective
                    additional = member_privs - group_effective
                    if not redundant:
                        level = RedundancyLevel.NONE
                        recommendation = "No action needed - member has unique privileges"
                    elif not additional:
                        level = RedundancyLevel.FULL
                        recommendation = f"REVOKE all privileges from {grant.principal} - fully covered by group"
                    else:
                        level = RedundancyLevel.PARTIAL
                        recommendation = f"Consider revoking {list(redundant)} from {grant.principal} - covered by group; keeping {list(additional)}"
                    results.append(RedundancyResult(
                        catalog_name=catalog_name, principal=grant.principal,
                        principal_type=grant.principal_type,
                        member_privileges=list(member_privs),
                        group_effective_privileges=list(group_effective),
                        redundant_privileges=list(redundant),
                        additional_privileges=list(additional),
                        redundancy_level=level, recommendation=recommendation
                    ))
                    if level != RedundancyLevel.NONE:
                        print(f"  \u26a0\ufe0f {catalog_name}: {grant.principal} has {level.value} redundancy")
                        print(f"      Member: {list(member_privs)}")
                        print(f"      Group covers: {list(redundant)}")
                        if additional:
                            print(f"      Additional (not covered): {list(additional)}")
            full_count = sum(1 for r in results if r.redundancy_level == RedundancyLevel.FULL)
            partial_count = sum(1 for r in results if r.redundancy_level == RedundancyLevel.PARTIAL)
            print(f"\n\u2705 Redundancy analysis complete:")
            print(f"   Full redundancy: {full_count} grant(s) - can be safely removed")
            print(f"   Partial redundancy: {partial_count} grant(s) - review recommended")
            return results

# Initialize redundancy detector
redundancy_detector = RedundancyDetector()
print("\u2705 Redundancy detector initialized")

In [0]:
if not PACKAGE_AVAILABLE:
    class RevokeScriptGenerator:
        """Generates copy-paste-ready REVOKE SQL from redundancy results."""
        
        @staticmethod
        def generate(redundancy_results: List[RedundancyResult],
                     include_partial: bool = False) -> str:
            lines: List[str] = []
            lines.append("-- " + "=" * 66)
            lines.append("-- AUTO-GENERATED REVOKE SCRIPT")
            lines.append(f"-- Generated: {datetime.now().isoformat()}")
            lines.append("-- Review carefully before executing!")
            lines.append("-- " + "=" * 66)
            lines.append("")
            full_count = 0
            partial_count = 0
            for r in redundancy_results:
                if r.redundancy_level == RedundancyLevel.FULL:
                    full_count += 1
                    privs = ", ".join(r.member_privileges)
                    principal = r.principal
                    if "@" in principal or " " in principal:
                        principal = f"`{principal}`"
                    lines.append(f"-- [FULL REDUNDANCY] {r.principal} on {r.catalog_name}")
                    lines.append(f"-- Member privileges {r.member_privileges} fully covered by group")
                    lines.append(f"REVOKE {privs} ON CATALOG `{r.catalog_name}` FROM {principal};")
                    lines.append("")
                elif r.redundancy_level == RedundancyLevel.PARTIAL and include_partial:
                    partial_count += 1
                    redundant_privs = ", ".join(r.redundant_privileges)
                    additional_privs = ", ".join(r.additional_privileges)
                    principal = r.principal
                    if "@" in principal or " " in principal:
                        principal = f"`{principal}`"
                    lines.append(f"-- [PARTIAL REDUNDANCY] {r.principal} on {r.catalog_name}")
                    lines.append(f"-- Redundant (covered by group): {r.redundant_privileges}")
                    lines.append(f"-- Additional (kept):            {r.additional_privileges}")
                    lines.append(f"-- Uncomment the line below after review:")
                    lines.append(f"-- REVOKE {redundant_privs} ON CATALOG `{r.catalog_name}` FROM {principal};")
                    lines.append("")
            if not full_count and not partial_count:
                lines.append("-- No redundant grants found - nothing to revoke.")
            else:
                lines.append("-- " + "-" * 66)
                lines.append(f"-- Summary: {full_count} full revoke(s), {partial_count} partial (commented out)")
                lines.append("-- " + "-" * 66)
            return "\n".join(lines)

revoke_generator = RevokeScriptGenerator()
print("\u2705 REVOKE script generator initialized")

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, BooleanType

class AuditResultBuilder:
    """Builds structured DataFrames from audit results."""
    
    def __init__(self, spark):
        self.spark = spark
    
    def build_permissions_df(self, catalog_grants: List[CatalogGrant]):
        """Create permissions table DataFrame."""
        rows = []
        for g in catalog_grants:
            rows.append(Row(
                catalog_name=g.catalog_name,
                workspace_name=g.workspace_name,
                principal=g.principal,
                principal_type=g.principal_type,
                privileges=", ".join(g.privileges),
                grant_source=g.grant_source.value,
                inherited_from=g.inherited_from or "",
                member_of_target=g.member_of_target
            ))
        
        if not rows:
            return self.spark.createDataFrame([], schema=StructType([
                StructField("catalog_name", StringType(), True),
                StructField("workspace_name", StringType(), True),
                StructField("principal", StringType(), True),
                StructField("principal_type", StringType(), True),
                StructField("privileges", StringType(), True),
                StructField("grant_source", StringType(), True),
                StructField("inherited_from", StringType(), True),
                StructField("member_of_target", BooleanType(), True)
            ]))
        
        return self.spark.createDataFrame(rows)
    
    def build_membership_df(self, group_node: GroupNode, all_members: Dict[str, List[GroupMember]]):
        """Create group membership tree DataFrame."""
        rows = []
        
        for user in all_members["users"]:
            rows.append(Row(
                member_id=user.id,
                display_name=user.display_name,
                member_type=user.member_type.value,
                email=user.email or "",
                application_id="",
                membership_path=" -> ".join(user.parent_groups)
            ))
        
        for sp in all_members["service_principals"]:
            rows.append(Row(
                member_id=sp.id,
                display_name=sp.display_name,
                member_type=sp.member_type.value,
                email="",
                application_id=sp.application_id or "",
                membership_path=" -> ".join(sp.parent_groups)
            ))
        
        if not rows:
            return self.spark.createDataFrame([], schema=StructType([
                StructField("member_id", StringType(), True),
                StructField("display_name", StringType(), True),
                StructField("member_type", StringType(), True),
                StructField("email", StringType(), True),
                StructField("application_id", StringType(), True),
                StructField("membership_path", StringType(), True)
            ]))
        
        return self.spark.createDataFrame(rows)
    
    def build_inheritance_df(self, catalog_grants: List[CatalogGrant]):
        """Create permission inheritance trace DataFrame."""
        rows = []
        for g in catalog_grants:
            rows.append(Row(
                catalog_name=g.catalog_name,
                principal=g.principal,
                principal_type=g.principal_type,
                privileges=", ".join(g.privileges),
                assignment_type=g.grant_source.value,
                source_group=g.inherited_from or "(direct)" if g.grant_source == GrantSource.UPSTREAM else "(self)" if g.grant_source == GrantSource.DIRECT else "(personal)"
            ))
        
        if not rows:
            return self.spark.createDataFrame([], schema=StructType([
                StructField("catalog_name", StringType(), True),
                StructField("principal", StringType(), True),
                StructField("principal_type", StringType(), True),
                StructField("privileges", StringType(), True),
                StructField("assignment_type", StringType(), True),
                StructField("source_group", StringType(), True)
            ]))
        
        return self.spark.createDataFrame(rows)
    
    def build_redundancy_df(self, redundancy_results: List[RedundancyResult]):
        """Create redundancy/overlap DataFrame."""
        rows = []
        for r in redundancy_results:
            rows.append(Row(
                catalog_name=r.catalog_name,
                principal=r.principal,
                principal_type=r.principal_type,
                member_privileges=", ".join(r.member_privileges),
                group_effective_privileges=", ".join(sorted(r.group_effective_privileges)),
                redundant_privileges=", ".join(r.redundant_privileges),
                additional_privileges=", ".join(r.additional_privileges),
                redundancy_level=r.redundancy_level.value,
                recommendation=r.recommendation
            ))
        
        if not rows:
            return self.spark.createDataFrame([], schema=StructType([
                StructField("catalog_name", StringType(), True),
                StructField("principal", StringType(), True),
                StructField("principal_type", StringType(), True),
                StructField("member_privileges", StringType(), True),
                StructField("group_effective_privileges", StringType(), True),
                StructField("redundant_privileges", StringType(), True),
                StructField("additional_privileges", StringType(), True),
                StructField("redundancy_level", StringType(), True),
                StructField("recommendation", StringType(), True)
            ]))
        
        return self.spark.createDataFrame(rows)
    
    def build_schema_grants_df(self, schema_grants: List[SchemaGrant]):
        """Create schema-level grants DataFrame."""
        rows = []
        for g in schema_grants:
            rows.append(Row(
                catalog_name=g.catalog_name,
                schema_name=g.schema_name,
                full_name=f"{g.catalog_name}.{g.schema_name}",
                workspace_name=g.workspace_name,
                principal=g.principal,
                principal_type=g.principal_type,
                privileges=", ".join(g.privileges),
                grant_source=g.grant_source.value,
                inherited_from=g.inherited_from or ""
            ))
        
        if not rows:
            return self.spark.createDataFrame([], schema=StructType([
                StructField("catalog_name", StringType(), True),
                StructField("schema_name", StringType(), True),
                StructField("full_name", StringType(), True),
                StructField("workspace_name", StringType(), True),
                StructField("principal", StringType(), True),
                StructField("principal_type", StringType(), True),
                StructField("privileges", StringType(), True),
                StructField("grant_source", StringType(), True),
                StructField("inherited_from", StringType(), True)
            ]))
        
        return self.spark.createDataFrame(rows)
    
    def build_summary(self, target_group: str, group_node: GroupNode,
                      all_members: Dict[str, List[GroupMember]],
                      catalog_grants: List[CatalogGrant],
                      schema_grants: List[SchemaGrant],
                      redundancy_results: List[RedundancyResult],
                      workspaces_scanned: int,
                      catalogs_scanned: int):
        """Build summary statistics dashboard."""
        
        def count_groups(node: GroupNode) -> int:
            count = len(node.nested_groups)
            for nested in node.nested_groups.values():
                count += count_groups(nested)
            return count
        
        nested_group_count = count_groups(group_node) if group_node else 0
        
        direct_grants = sum(1 for g in catalog_grants if g.grant_source == GrantSource.DIRECT)
        upstream_grants = sum(1 for g in catalog_grants if g.grant_source == GrantSource.UPSTREAM)
        member_direct_grants = sum(1 for g in catalog_grants if g.grant_source == GrantSource.MEMBER_DIRECT)
        
        full_redundant = sum(1 for r in redundancy_results if r.redundancy_level == RedundancyLevel.FULL)
        partial_redundant = sum(1 for r in redundancy_results if r.redundancy_level == RedundancyLevel.PARTIAL)
        
        member_grant_counts = {}
        for g in catalog_grants:
            if g.member_of_target:
                member_grant_counts[g.principal] = member_grant_counts.get(g.principal, 0) + 1
        
        top_members = sorted(member_grant_counts.items(), key=lambda x: -x[1])[:10]
        
        return {
            "target_group": target_group,
            "total_users": len(all_members["users"]),
            "total_service_principals": len(all_members["service_principals"]),
            "nested_groups": nested_group_count,
            "workspaces_scanned": workspaces_scanned,
            "catalogs_scanned": catalogs_scanned,
            "total_catalog_grants": len(catalog_grants),
            "direct_grants": direct_grants,
            "upstream_grants": upstream_grants,
            "member_direct_grants": member_direct_grants,
            "schema_grants": len(schema_grants),
            "table_grants": 0,  # set by caller after table scan
            "full_redundancy_count": full_redundant,
            "partial_redundancy_count": partial_redundant,
            "top_members_by_grants": top_members
        }
    
    def print_summary_dashboard(self, summary: Dict):
        """Print a formatted summary dashboard."""
        print("\n" + "="*70)
        print("                    GROUP AUDIT SUMMARY DASHBOARD")
        print("="*70)
        print(f"\n🎯 Target Group: {summary['target_group']}")
        print("\n👥 MEMBERSHIP:")
        print(f"   ├─ Users: {summary['total_users']}")
        print(f"   ├─ Service Principals: {summary['total_service_principals']}")
        print(f"   └─ Nested Groups: {summary['nested_groups']}")
        print("\n🔍 SCAN COVERAGE:")
        print(f"   ├─ Workspaces Scanned: {summary['workspaces_scanned']}")
        print(f"   └─ Catalogs Audited: {summary['catalogs_scanned']}")
        print("\n📊 CATALOG PERMISSIONS:")
        print(f"   ├─ Total Grants Found: {summary['total_catalog_grants']}")
        print(f"   ├─ Direct (group itself): {summary['direct_grants']}")
        print(f"   ├─ Upstream (inherited from parent): {summary['upstream_grants']}")
        print(f"   └─ Member Direct (personal grants): {summary['member_direct_grants']}")
        if summary.get('schema_grants', 0) > 0:
            print(f"\n📁 SCHEMA PERMISSIONS: {summary['schema_grants']} grant(s)")
        if summary.get('table_grants', 0) > 0:
            print(f"\n📝 TABLE / VIEW PERMISSIONS: {summary['table_grants']} grant(s)")
        print("\n⚠️ REDUNDANCY ANALYSIS:")
        print(f"   ├─ Fully Redundant Grants: {summary['full_redundancy_count']} (can be revoked)")
        print(f"   └─ Partially Redundant: {summary['partial_redundancy_count']} (review recommended)")
        if summary['top_members_by_grants']:
            print("\n🏆 TOP MEMBERS BY PERSONAL GRANTS:")
            for i, (member, count) in enumerate(summary['top_members_by_grants'], 1):
                print(f"   {i}. {member}: {count} grant(s)")
        print("\n" + "="*70)

# Initialize result builder
result_builder = AuditResultBuilder(spark)

print("✅ Output builder initialized")

In [0]:
# ============================================================================
# MAIN EXECUTION - Run the complete group audit
# ============================================================================
from pyspark.sql import functions as F

def run_audit():
    """Execute the complete group audit workflow."""
    
    # Validate configuration
    if not CLIENT_ID or not CLIENT_SECRET or not ACCOUNT_ID:
        raise ValueError("Please configure CLIENT_ID, CLIENT_SECRET, and ACCOUNT_ID")
    if not TARGET_GROUP_NAME:
        raise ValueError("Please configure TARGET_GROUP_NAME")
    
    print("\n" + "="*70)
    print("          DATABRICKS GROUP AUDIT TOOL")
    print("="*70)
    run_ts = datetime.now().isoformat()
    print(f"\nStarting audit for group: {TARGET_GROUP_NAME}")
    print(f"Timestamp: {run_ts}")
    
    # ---- Step 1: Resolve group membership --------------------------------
    print("\n" + "-"*50)
    print("STEP 1: Resolving Group Membership")
    print("-"*50)
    group_node = group_resolver.resolve_group(TARGET_GROUP_NAME)
    if not group_node:
        raise ValueError(f"Group '{TARGET_GROUP_NAME}' not found in account")
    all_members = group_resolver.get_all_members_flat(group_node)
    print(f"\n\u2705 Found {len(all_members['users'])} users and {len(all_members['service_principals'])} service principals")
    
    # ---- Step 2: Discover workspaces -------------------------------------
    print("\n" + "-"*50)
    print("STEP 2: Discovering Workspaces")
    print("-"*50)
    workspaces = workspace_discovery.discover(WORKSPACE_URLS)
    if not workspaces:
        raise ValueError("No workspaces found to scan")
    
    # ---- Step 3: Scan catalog permissions ---------------------------------
    print("\n" + "-"*50)
    print("STEP 3: Scanning Catalog Permissions")
    print("-"*50)
    catalog_grants = catalog_scanner.scan_all_workspaces(
        workspaces, TARGET_GROUP_NAME, group_node, all_members
    )
    
    # Pre-compute shared lookups for schema/table steps
    # (compatible with both package public name and notebook private name)
    _upstream_fn = getattr(catalog_scanner, 'get_groups_containing_target',
                           getattr(catalog_scanner, '_get_groups_containing_target', None))
    upstream_groups = _upstream_fn(TARGET_GROUP_NAME) if _upstream_fn else {}
    accessible_catalogs = set()
    for g in catalog_grants:
        if g.grant_source in [GrantSource.DIRECT, GrantSource.UPSTREAM]:
            accessible_catalogs.add((g.catalog_name, g.workspace_name, g.workspace_url))
    
    def _make_ws(ws_name, ws_url):
        return WorkspaceInfo(workspace_id="scan", deployment_name=ws_name,
                             workspace_name=ws_name, workspace_url=ws_url,
                             cloud=CLOUD_PROVIDER.upper(), region="")
    
    # ---- Step 4: Optional schema scanning ---------------------------------
    schema_grants = []
    if SCAN_SCHEMAS or SCAN_TABLES:
        print("\n" + "-"*50)
        print("STEP 4: Scanning Schema Permissions")
        print("-"*50)
        for catalog_name, ws_name, ws_url in accessible_catalogs:
            print(f"\n  Scanning schemas in {catalog_name}...")
            ws = _make_ws(ws_name, ws_url)
            schema_grants.extend(
                schema_scanner.scan_schemas(ws, catalog_name, TARGET_GROUP_NAME, all_members, upstream_groups)
            )
        print(f"\n\u2705 Found {len(schema_grants)} schema-level grant(s)")
    else:
        print("\n\u23e9 Schema scanning disabled")
    
    # ---- Step 4b: Optional table scanning ---------------------------------
    table_grants = []
    if SCAN_TABLES:
        print("\n" + "-"*50)
        print("STEP 4b: Scanning Table / View Permissions")
        print("-"*50)
        for catalog_name, ws_name, ws_url in accessible_catalogs:
            ws = _make_ws(ws_name, ws_url)
            schemas = schema_scanner.get_schemas(ws, catalog_name)
            for schema in schemas:
                schema_name = schema.get("name", "")
                print(f"    Scanning tables in {catalog_name}.{schema_name}...")
                table_grants.extend(
                    table_scanner.scan_tables(ws, catalog_name, schema_name,
                                              TARGET_GROUP_NAME, all_members, upstream_groups)
                )
        print(f"\n\u2705 Found {len(table_grants)} table-level grant(s)")
    else:
        print("\n\u23e9 Table scanning disabled")
    
    # ---- Step 5: Detect redundancy ----------------------------------------
    print("\n" + "-"*50)
    print("STEP 5: Detecting Redundant Grants")
    print("-"*50)
    redundancy_results = redundancy_detector.detect_redundancy(catalog_grants, TARGET_GROUP_NAME)
    
    # ---- Step 5b: Generate REVOKE script ----------------------------------
    print("\n" + "-"*50)
    print("STEP 5b: Generating REVOKE Script")
    print("-"*50)
    global revoke_sql
    revoke_sql = revoke_generator.generate(redundancy_results, include_partial=True)
    print(revoke_sql)
    
    # ---- Step 6: Build output DataFrames ----------------------------------
    print("\n" + "-"*50)
    print("STEP 6: Building Output DataFrames")
    print("-"*50)
    
    global df_permissions, df_membership, df_inheritance, df_redundancy, df_schema_grants, df_table_grants, audit_summary
    
    df_permissions  = result_builder.build_permissions_df(catalog_grants)
    df_membership   = result_builder.build_membership_df(group_node, all_members)
    df_inheritance  = result_builder.build_inheritance_df(catalog_grants)
    df_redundancy   = result_builder.build_redundancy_df(redundancy_results)
    df_schema_grants = result_builder.build_schema_grants_df(schema_grants)
    
    # Table grants DF
    tbl_rows = []
    for g in table_grants:
        tbl_rows.append(Row(
            catalog_name=g.catalog_name, schema_name=g.schema_name,
            table_name=g.table_name, full_name=g.full_name,
            table_type=g.table_type, workspace_name=g.workspace_name,
            principal=g.principal, principal_type=g.principal_type,
            privileges=", ".join(g.privileges),
            grant_source=g.grant_source.value,
            inherited_from=g.inherited_from or ""
        ))
    if tbl_rows:
        df_table_grants = spark.createDataFrame(tbl_rows)
    else:
        from pyspark.sql.types import StructType, StructField, StringType
        df_table_grants = spark.createDataFrame([], schema=StructType([
            StructField(c, StringType(), True) for c in
            ["catalog_name","schema_name","table_name","full_name","table_type",
             "workspace_name","principal","principal_type","privileges",
             "grant_source","inherited_from"]
        ]))
    
    audit_summary = result_builder.build_summary(
        TARGET_GROUP_NAME, group_node, all_members,
        catalog_grants, schema_grants, redundancy_results,
        len(workspaces), len(catalog_scanner._scanned_catalogs)
    )
    audit_summary["table_grants"] = len(table_grants)
    
    print("\u2705 DataFrames created:")
    print("   \u2022 df_permissions    - Catalog permission grants")
    print("   \u2022 df_membership     - Group membership tree")
    print("   \u2022 df_inheritance    - Permission inheritance trace")
    print("   \u2022 df_redundancy     - Redundancy analysis results")
    print("   \u2022 df_schema_grants  - Schema-level grants")
    print("   \u2022 df_table_grants   - Table/view-level grants")
    print("   \u2022 revoke_sql        - REVOKE cleanup script (string)")
    
    # ---- Step 7: Optional Delta export ------------------------------------
    if EXPORT_DELTA_PATH:
        print("\n" + "-"*50)
        print("STEP 7: Exporting to Delta")
        print("-"*50)
        _ts = F.lit(run_ts)
        _grp = F.lit(TARGET_GROUP_NAME)
        
        for name, df in [
            ("permissions", df_permissions), ("membership", df_membership),
            ("inheritance", df_inheritance), ("redundancy", df_redundancy),
            ("schema_grants", df_schema_grants), ("table_grants", df_table_grants)
        ]:
            target = f"{EXPORT_DELTA_PATH}/{name}"
            (df
             .withColumn("audit_timestamp", _ts)
             .withColumn("target_group", _grp)
             .write.format("delta")
             .mode("append")
             .option("mergeSchema", "true")
             .save(target))
            print(f"  \u2705 {name} \u2192 {target}")
        
        print(f"\n\u2705 All DataFrames exported to {EXPORT_DELTA_PATH}")
    else:
        print("\n\u23e9 Delta export disabled (set export_delta_path to enable)")
    
    # ---- Step 8: Summary dashboard ----------------------------------------
    result_builder.print_summary_dashboard(audit_summary)
    
    return {
        "permissions": df_permissions,
        "membership": df_membership,
        "inheritance": df_inheritance,
        "redundancy": df_redundancy,
        "schema_grants": df_schema_grants,
        "table_grants": df_table_grants,
        "revoke_sql": revoke_sql,
        "summary": audit_summary
    }

# Run the audit
results = run_audit()

In [0]:
# ============================================================================
# PRINCIPAL AUDIT — reverse lookup from a user / SP / group
#
# Set the "principal_identifier" widget below to an email, SP app-ID,
# SP display name, or group display name and run this cell.
# ============================================================================
try:
    dbutils.widgets.text("principal_identifier", "", "Principal to audit (email / SP / group)")
except Exception:
    pass

PRINCIPAL_ID = dbutils.widgets.get("principal_identifier").strip()

if PRINCIPAL_ID:
    if PACKAGE_AVAILABLE:
        from databricks_group_audit import PrincipalAuditor
        _pa = PrincipalAuditor(api_client, workspace_discovery=workspace_discovery, cloud_provider=CLOUD_PROVIDER)
    else:
        # ----- Inline PrincipalAuditor when package is not installed -----
        from collections import deque as _deque

        class PrincipalAuditor:
            def __init__(self, api, ws_disc, cloud):
                self.api = api
                self.ws_disc = ws_disc
                self.cloud = cloud.upper()

            def find_principal(self, identifier):
                identifier = identifier.strip()
                if "@" in identifier:
                    try:
                        resp = self.api.account_api("GET", "/scim/v2/Users", params={"filter": f'emails.value eq "{identifier}"'})
                        for u in resp.get("Resources", []):
                            return "USER", u["id"], u.get("displayName", identifier)
                    except Exception:
                        pass
                for filt in (f'applicationId eq "{identifier}"', f'displayName eq "{identifier}"'):
                    try:
                        resp = self.api.account_api("GET", "/scim/v2/ServicePrincipals", params={"filter": filt})
                        for sp in resp.get("Resources", []):
                            return "SERVICE_PRINCIPAL", sp["id"], sp.get("displayName", identifier)
                    except Exception:
                        pass
                try:
                    resp = self.api.account_api("GET", "/scim/v2/Groups", params={"filter": f'displayName eq "{identifier}"'})
                    for g in resp.get("Resources", []):
                        return "GROUP", g["id"], g.get("displayName", identifier)
                except Exception:
                    pass
                raise ValueError(f"Principal '{identifier}' not found as user, SP, or group.")

            def resolve_group_memberships(self, pid, ptype, pname):
                all_groups = self.api.scim_list_all("Groups")
                id_to_name = {}
                child_to_parents = {}
                for g in all_groups:
                    gid = g.get("id", "")
                    gname = g.get("displayName", "")
                    id_to_name[gid] = gname
                    for m in g.get("members", []):
                        cid = m.get("value", "")
                        if cid:
                            child_to_parents.setdefault(cid, set()).add(gid)
                direct_ids = child_to_parents.get(pid, set())
                memberships = []
                visited = set()
                queue = _deque()
                for did in direct_ids:
                    queue.append((did, [pname, id_to_name.get(did, did)], True))
                while queue:
                    gid, path, is_direct = queue.popleft()
                    if gid in visited:
                        continue
                    visited.add(gid)
                    gname = id_to_name.get(gid, gid)
                    memberships.append({"group_id": gid, "group_name": gname, "path": list(path), "is_direct": is_direct})
                    for parent_id in child_to_parents.get(gid, set()):
                        if parent_id not in visited:
                            queue.append((parent_id, path + [id_to_name.get(parent_id, parent_id)], False))
                return memberships, id_to_name

            def get_workspace_assignments(self, workspaces, pid, group_ids, id_to_name):
                roles = []
                relevant = group_ids | {pid}
                for ws in workspaces:
                    try:
                        resp = self.api.account_api("GET", f"/workspaces/{ws.workspace_id}/permissionassignments")
                        for pa in resp.get("permission_assignments", []):
                            p_id = str(pa.get("principal", {}).get("principal_id", ""))
                            if p_id not in relevant:
                                continue
                            for perm in pa.get("permissions", []):
                                via = id_to_name.get(p_id, p_id)
                                roles.append({"workspace_id": ws.workspace_id, "workspace_name": ws.workspace_name,
                                              "workspace_url": ws.workspace_url, "permission_level": perm,
                                              "via_group": via if p_id != pid else "(direct)", "via_group_id": p_id})
                    except Exception as exc:
                        print(f"  \u26a0\ufe0f Failed to get assignments for {ws.workspace_name}: {exc}")
                return roles

            def scan_permissions(self, ws_roles, pname, group_names):
                perms = []
                relevant = group_names | {pname}
                seen_ws = set()
                scanned_cats = set()
                for role in ws_roles:
                    ws_url = role["workspace_url"] if isinstance(role, dict) else role.workspace_url
                    ws_name = role["workspace_name"] if isinstance(role, dict) else role.workspace_name
                    if ws_url in seen_ws:
                        continue
                    seen_ws.add(ws_url)
                    try:
                        catalogs = self.api.workspace_api(ws_url, "GET", "/api/2.1/unity-catalog/catalogs").get("catalogs", [])
                    except Exception:
                        continue
                    for cat in catalogs:
                        cname = cat.get("name", "")
                        if cname in scanned_cats:
                            continue
                        scanned_cats.add(cname)
                        try:
                            grants = self.api.workspace_api(ws_url, "GET", f"/api/2.1/unity-catalog/permissions/catalog/{cname}").get("privilege_assignments", [])
                        except Exception:
                            continue
                        for g in grants:
                            principal = g.get("principal", "")
                            if principal in relevant or principal.replace("`", "") in relevant or principal.lower() in {n.lower() for n in relevant}:
                                perms.append({"securable_type": "CATALOG", "securable_name": cname,
                                              "privileges": g.get("privileges", []), "via_group": principal,
                                              "workspace_name": ws_name, "workspace_url": ws_url})
                return perms

        _pa = PrincipalAuditor(api_client, workspace_discovery, CLOUD_PROVIDER)

    # ---- Run the audit ----
    print("\n" + "="*70)
    print("          PRINCIPAL AUDIT (REVERSE LOOKUP)")
    print("="*70)
    print(f"\nAuditing: {PRINCIPAL_ID}")

    try:
        if PACKAGE_AVAILABLE:
            pa_result = _pa.audit(
                identifier=PRINCIPAL_ID,
                explicit_workspace_urls=WORKSPACE_URLS,
                scan_schemas=SCAN_SCHEMAS,
                scan_tables=SCAN_TABLES,
            )
            _groups = pa_result.groups
            _ws_roles = pa_result.workspace_roles
            _perms = pa_result.permissions
            _dead = pa_result.dead_end_groups
            _pname = pa_result.principal_name
            _ptype = pa_result.principal_type
        else:
            _ptype, _pid, _pname = _pa.find_principal(PRINCIPAL_ID)
            print(f"\u2705 Found: {_pname} ({_ptype})")
            _memberships, _id_map = _pa.resolve_group_memberships(_pid, _ptype, _pname)
            _group_ids = {m["group_id"] for m in _memberships}
            _group_names = {m["group_name"] for m in _memberships}
            _workspaces = workspace_discovery.discover(WORKSPACE_URLS)
            _ws_roles = _pa.get_workspace_assignments(_workspaces, _pid, _group_ids, _id_map)
            _groups_with_access = {r["via_group_id"] for r in _ws_roles if r["via_group"] != "(direct)"}
            _dead = [m["group_name"] for m in _memberships if m["group_id"] not in _groups_with_access]
            _perms = _pa.scan_permissions(_ws_roles, _pname, _group_names)
            _groups = _memberships

        # ---- Display results ----
        print(f"\n\U0001f464 Principal: {_pname} ({_ptype if isinstance(_ptype, str) else _ptype})")

        print(f"\n\U0001f4c2 Group Memberships ({len(_groups)}):")
        for g in _groups:
            name = g.group_name if hasattr(g, 'group_name') else g["group_name"]
            direct = g.is_direct if hasattr(g, 'is_direct') else g["is_direct"]
            path = g.path if hasattr(g, 'path') else g["path"]
            tag = "\u2b50 direct" if direct else "\u21b3 transitive"
            print(f"  {tag}: {name}")
            print(f"         path: {' \u2192 '.join(path)}")

        print(f"\n\U0001f4bc Workspace Access ({len(_ws_roles)}):")
        for r in _ws_roles:
            ws_name = r.workspace_name if hasattr(r, 'workspace_name') else r["workspace_name"]
            perm = r.permission_level if hasattr(r, 'permission_level') else r["permission_level"]
            via = r.via_group if hasattr(r, 'via_group') else r["via_group"]
            print(f"  \u2705 {ws_name}: {perm} (via {via})")

        if _dead:
            print(f"\n\u26a0\ufe0f Dead-End Groups (no workspace access): {len(_dead)}")
            for dg in _dead:
                print(f"  \u274c {dg}")

        print(f"\n\U0001f512 UC Permissions ({len(_perms)}):")
        for p in _perms:
            stype = p.securable_type if hasattr(p, 'securable_type') else p["securable_type"]
            sname = p.securable_name if hasattr(p, 'securable_name') else p["securable_name"]
            privs = p.privileges if hasattr(p, 'privileges') else p["privileges"]
            via = p.via_group if hasattr(p, 'via_group') else p["via_group"]
            ws = p.workspace_name if hasattr(p, 'workspace_name') else p["workspace_name"]
            print(f"  [{stype}] {sname}")
            print(f"    privileges: {', '.join(privs)}")
            print(f"    via: {via} @ {ws}")

        # Build DataFrames
        from pyspark.sql import Row
        from pyspark.sql.types import StructType, StructField, StringType, BooleanType

        _g_rows = []
        for g in _groups:
            name = g.group_name if hasattr(g, 'group_name') else g["group_name"]
            direct = g.is_direct if hasattr(g, 'is_direct') else g["is_direct"]
            path = g.path if hasattr(g, 'path') else g["path"]
            _g_rows.append(Row(group_name=name, is_direct=direct, path=" \u2192 ".join(path)))
        df_principal_groups = spark.createDataFrame(_g_rows) if _g_rows else spark.createDataFrame([], StructType([StructField("group_name", StringType()), StructField("is_direct", BooleanType()), StructField("path", StringType())]))

        _r_rows = []
        for r in _ws_roles:
            _r_rows.append(Row(
                workspace_name=r.workspace_name if hasattr(r, 'workspace_name') else r["workspace_name"],
                permission_level=r.permission_level if hasattr(r, 'permission_level') else r["permission_level"],
                via_group=r.via_group if hasattr(r, 'via_group') else r["via_group"],
            ))
        df_principal_ws = spark.createDataFrame(_r_rows) if _r_rows else spark.createDataFrame([], StructType([StructField("workspace_name", StringType()), StructField("permission_level", StringType()), StructField("via_group", StringType())]))

        _p_rows = []
        for p in _perms:
            _p_rows.append(Row(
                securable_type=p.securable_type if hasattr(p, 'securable_type') else p["securable_type"],
                securable_name=p.securable_name if hasattr(p, 'securable_name') else p["securable_name"],
                privileges=", ".join(p.privileges if hasattr(p, 'privileges') else p["privileges"]),
                via_group=p.via_group if hasattr(p, 'via_group') else p["via_group"],
                workspace_name=p.workspace_name if hasattr(p, 'workspace_name') else p["workspace_name"],
            ))
        df_principal_perms = spark.createDataFrame(_p_rows) if _p_rows else spark.createDataFrame([], StructType([StructField("securable_type", StringType()), StructField("securable_name", StringType()), StructField("privileges", StringType()), StructField("via_group", StringType()), StructField("workspace_name", StringType())]))

        print("\n\u2705 DataFrames created:")
        print("   \u2022 df_principal_groups  - Group memberships")
        print("   \u2022 df_principal_ws      - Workspace roles")
        print("   \u2022 df_principal_perms   - UC permissions")

    except ValueError as exc:
        print(f"\n\u274c {exc}")
    except Exception as exc:
        print(f"\n\u274c Unexpected error: {exc}")
        raise

else:
    print("\u23e9 Principal audit skipped (set 'principal_identifier' widget to run)")

In [0]:
# Display the principal's group memberships
if PRINCIPAL_ID and 'df_principal_groups' in dir():
    print("\U0001f4c2 Principal Group Memberships")
    display(df_principal_groups)
else:
    print("\u2139\ufe0f Run the Principal Audit cell first (set principal_identifier widget)")

In [0]:
# Display the principal's workspace access
if PRINCIPAL_ID and 'df_principal_ws' in dir():
    print("\U0001f4bc Principal Workspace Access")
    display(df_principal_ws)
else:
    print("\u2139\ufe0f Run the Principal Audit cell first (set principal_identifier widget)")

In [0]:
# Display the principal's UC permissions
if PRINCIPAL_ID and 'df_principal_perms' in dir():
    print("\U0001f512 Principal UC Permissions")
    display(df_principal_perms)
else:
    print("\u2139\ufe0f Run the Principal Audit cell first (set principal_identifier widget)")

In [0]:
# Display the catalog permissions table
print("📋 Catalog Permissions (Direct, Upstream, and Member Direct grants)")
display(df_permissions)

In [0]:
# Display the group membership tree
print("👥 Group Membership (All users and service principals with inheritance paths)")
display(df_membership)

In [0]:
# Display the permission inheritance trace
print("🔗 Permission Inheritance Trace (How each permission was acquired)")
display(df_inheritance)

In [0]:
# Display redundancy analysis results
print("⚠️ Redundancy Analysis (Overlapping grants that may be revoked)")
display(df_redundancy)

In [0]:
# Display schema-level grants (only populated if SCAN_SCHEMAS=true)
if df_schema_grants.count() > 0:
    print("📁 Schema-Level Grants")
    display(df_schema_grants)
else:
    print("ℹ️ No schema grants to display (schema scanning may be disabled or no grants found)")

In [0]:
# Display table-level grants (only populated if SCAN_TABLES=true)
if df_table_grants.count() > 0:
    print("📝 Table / View Level Grants")
    display(df_table_grants)
else:
    print("ℹ️ No table grants to display (table scanning may be disabled or no grants found)")

In [0]:
# Display the auto-generated REVOKE SQL script
print("📜 Copy-paste-ready REVOKE script (review before executing!):")
print()
print(revoke_sql)